In [ ]:
# Create a sample DataFrame
data = [(101, "Standard", 5.99),
(102, "Express", 15.99),
(103, "Standard", 5.99),
(104, "Standard", 5.99),
(105, "Overnight", 25.99)
]
columns = ["OrderID", "ShippingMethod", "ShippingCost"]
df = spark.createDataFrame(data, columns)
# Write the DataFrame to a Delta table named 'orders'
df.write.mode("overwrite").format("delta").saveAsTable("orders")
print("Table 'orders' created successfully.")

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can't be run because you've hit spark overall capacity compute limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [1]:
%%sql
UPDATE orders
SET ShippingMethod = 'Express', ShippingCost = 15.99
WHERE ShippingMethod = 'Standard';
Verify that the error occurred. Run a SELECT query again:

SELECT * FROM orders;

StatementMeta(, , -1, Finished, , Finished, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can't be run because you've hit spark overall capacity compute limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [ ]:
# Compare Version 0 (before error) with Version 1 (after error)
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table("orders")
df_v1 = spark.read.format("delta").option("versionAsOf", 1).table("orders")
print("Version 0 - Correct Data:")
df_v0.show()
print("Version 1 - Erroneous Data:")
df_v1.show()

In [ ]:
# Simulate an accidental DELETE (optional scenario)
spark.sql("DELETE FROM orders WHERE ShippingMethod = 'Overnight'")
# Validate the missing row
print("After DELETE (rows missing):")
spark.sql("SELECT * FROM orders").show()
# Recover deleted row using Time Travel (Version 1 was before DELETE)
df_v1 = spark.read.format("delta").option("versionAsOf", 1).table("orders")
print("Recovered Data from Version 1:")
df_v1.show()

In [ ]:
# Restore data from Version 0 into a new table
df_restored = spark.read.format("delta").option("versionAsOf", 0).table("orders")
df_restored.write.format("delta").mode("overwrite").saveAsTable("orders_restored")
print("Restored table created as 'orders_restored'")
spark.sql("SELECT * FROM orders_restored").show()

In [ ]:
# Restore using timestamp (replace with actual timestamp from DESCRIBE HISTORY)
df_time = spark.read.format("delta").option("timestampAsOf", "2025-10-01T12:00:00.000Z").table("orders")
df_time.write.format("delta").mode("overwrite").saveAsTable("orders_restored_timestamp")
print("Restored table created using timestamp-based recovery")
spark.sql("SELECT * FROM orders_restored_timestamp").show()

In [ ]:
# Load the table as it existed before the DELETE (for example, version 2)
df_before_delete = spark.read.format("delta").option("versionAsOf", 2).table("orders")
# Display the data before the DELETE occurred
df_before_delete.show()
# Restore the correct version to a new table
df_before_delete.write.format("delta").mode("overwrite").saveAsTable("orders_restored")
print("Restored table created successfully.")

In [ ]:
# Load the table as it existed before a specific timestamp
df_recovered = spark.read.format("delta") \
.option("timestampAsOf", "2025-10-04T14:59:00Z") \
.table("orders")
# Display the data from that timestamp
df_recovered.show()